# Chapter 3 - Lab 8: Mistral with a manual tool loop

**Goal**: use **Mistral**'s API client with function calling and write the agent loop **by hand**. This is exactly what the frameworks in Labs 1-6 do for you.

This is the same reference task used across every framework lab in Chapter 3:
**compare the Zenith X1 smartphone (`ZX1`) and the FrostLine refrigerator (`FL-340`) on contact rate**
(support tickets per 100 delivered orders) and write a short memo for the Head of Customer Care.
Comparing all frameworks on the *same* task makes differences in API style, abstractions and ergonomics easy to spot.

*Finance edition*: compare Apple and JPMorgan on P/E. Contact rate plays the role of P/E: a ratio
that lets a high-volume and a low-volume product be compared fairly (ZX1 = 15.0, FL-340 = 36.0).

**Tech stack**:
* **Mistral** (`mistralai`) - `client.chat.complete` with `tools`.
* Model `mistral-medium-latest`.

## 1. Install packages

In [ ]:
%pip install -q mistralai pydantic python-dotenv

## 2. Set up the API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['MISTRAL_API_KEY'] = userdata.get('MISTRAL_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('MISTRAL_API_KEY'), 'Set MISTRAL_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared task setup

The task, tools, dataset and prompts live in `common.py` (next to this notebook in the repository).

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%203/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import (
    get_support_data,
    compute_contact_rate,
    support_data,
    system_message,
    input_message,
)

print("Tools loaded. Reference task:")
print(" ", input_message)

## 4. Tool schemas (OpenAI-compatible JSON)

In [ ]:
tools = [
    {"type": "function", "function": {
        "name": "get_support_data",
        "description": "Get support tickets and delivered orders for a product line",
        "parameters": {"type": "object",
                       "properties": {"product_line": {"type": "string", "description": "e.g. ZX1, FL-340"}},
                       "required": ["product_line"]}}},
    {"type": "function", "function": {
        "name": "compute_contact_rate",
        "description": "Compute contact rate (tickets per 100 orders)",
        "parameters": {"type": "object",
                       "properties": {"tickets": {"type": "integer"}, "orders": {"type": "integer"}},
                       "required": ["tickets", "orders"]}}},
]

## 5. Tool executor (a dispatcher from tool name to Python function)

In [ ]:
def execute_tool(name: str, args: dict) -> str:
    if name == "get_support_data":
        r = get_support_data(args["product_line"])
        return f"{args['product_line']}: tickets={r.tickets}, orders={r.orders}"
    if name == "compute_contact_rate":
        return f"Contact rate: {compute_contact_rate(args['tickets'], args['orders'])} per 100 orders"
    return f"Unknown tool: {name}"

## 6. The manual agent loop (capped at 5 iterations)

In [ ]:
import json, os
from mistralai.client import Mistral

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
messages = [{"role": "system", "content": system_message},
            {"role": "user", "content": input_message}]

for iteration in range(5):
    response = client.chat.complete(model="mistral-medium-latest", messages=messages,
                                    tools=tools, tool_choice="auto")
    msg = response.choices[0].message
    messages.append({"role": "assistant", "content": msg.content, "tool_calls": msg.tool_calls})
    if not msg.tool_calls:
        break
    for tc in msg.tool_calls:
        result = execute_tool(tc.function.name, json.loads(tc.function.arguments))
        print(f"[{tc.function.name}] -> {result}")
        messages.append({"role": "tool", "name": tc.function.name, "content": result, "tool_call_id": tc.id})

print("\n[Memo]\n" + str(msg.content))

## 7. Results
Twenty lines of loop code: **send -> if tool calls, execute and append -> repeat until no tool calls**.
Every framework in this chapter wraps this loop and adds tracing, retries, memory and guardrails.
Chapter 12 (`mini_care_agent.py`) revisits this loop to explain what a production *harness* adds.